In [11]:
%pip install pyshacl rdflib

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 21.5 MB/s  0:00:00
   ---------------------------------------- 0.0/615.4 kB ? eta -:--:--
   ---------------------------------------- 615.4/615.4 kB 20.4 MB/s  0:00:00
Using cached prettytable-3.19.0-py3-none-any.whl (37 kB)

   ---------------------------------------- 0/5 [rdflib]
   ---------------------------------------- 0/5 [rdflib]
   ---------------------------------------- 0/5 [rdflib]
   ---------------------------------------- 0/5 [rdflib]
   ---------------------------------------- 0/5 [rdflib]
   ---------------------------------------- 0/5 [rdflib]
   ---------------------------------------- 0/5 [rdflib]
   -------- ------------------------------- 1/5 [prettytable]
   ---------------- ----------------------- 2/5 [html5rdf]
   ---------------- ----------------------- 2/5 [html5rdf]
   ------------------------ --------------- 3/5 [owlrl]
   ------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Génération du faux patient 
patient_data = """@prefix fhir: <http://hl7.org/fhir/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.com/patient/> .

ex:Patient1 a fhir:Patient ;
    fhir:Patient.birthDate [ fhir:value "1990-01-01"^^xsd:date ] ;
    fhir:Patient.deceasedDateTime [ fhir:value "1985-05-15"^^xsd:date ] .
"""
with open("patient_invalide.ttl", "w", encoding="utf-8") as f:
    f.write(patient_data)

# Génération de la règle clinique en SHACL
shacl_rules = """@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix fhir: <http://hl7.org/fhir/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.com/patient/> .

ex:PatientDeathShape a sh:NodeShape ;
    sh:targetClass fhir:Patient ;
    sh:sparql [
        a sh:SPARQLConstraint ;
        sh:message "ERREUR CLINIQUE : La date de décès est antérieure à la date de naissance." ;
        sh:select '''
            PREFIX fhir: <http://hl7.org/fhir/>
            SELECT $this ?birth ?death
            WHERE {
                $this fhir:Patient.birthDate ?bNode .
                ?bNode fhir:value ?birth .
                $this fhir:Patient.deceasedDateTime ?dNode .
                ?dNode fhir:value ?death .
                FILTER (?death < ?birth)
            }
        ''' ;
    ] .
"""
with open("shapes.ttl", "w", encoding="utf-8") as f:
    f.write(shacl_rules)

print(" Fichiers locaux créés ")

 Fichiers locaux créés 


In [3]:
from pyshacl import validate
from rdflib import Graph

data_graph = Graph().parse("patient_invalide.ttl", format="turtle")
shacl_graph = Graph().parse("shapes.ttl", format="turtle")

conforms, results_graph, results_text = validate(
    data_graph,
    shacl_graph=shacl_graph,
    data_graph_format="turtle",
    shacl_graph_format="turtle",
    inference="rdfs",
    debug=False,
    serialize_report_graph=True
)

if conforms:
    print(" RÉSULTAT : Le dossier patient est cliniquement valide.")
else:
    print(" RÉSULTAT : Incohérence détectée !")
    print("-" * 50)
    print(results_text)

 RÉSULTAT : Incohérence détectée !
--------------------------------------------------
Validation Report
Conforms: False
Results (1):
Constraint Violation in SPARQLConstraintComponent (http://www.w3.org/ns/shacl#SPARQLConstraintComponent):
	Severity: sh:Violation
	Source Shape: ex:PatientDeathShape
	Focus Node: ex:Patient1
	Value Node: ex:Patient1
	Source Constraint: [ rdf:type sh:SPARQLConstraint ; sh:message Literal("ERREUR CLINIQUE : La date de décès est antérieure à la date de naissance.") ; sh:select Literal("
            PREFIX fhir: <http://hl7.org/fhir/>
            SELECT $this ?birth ?death
            WHERE {
                $this fhir:Patient.birthDate ?bNode .
                ?bNode fhir:value ?birth .
                $this fhir:Patient.deceasedDateTime ?dNode .
                ?dNode fhir:value ?death .
                FILTER (?death < ?birth)
            }
        ") ]
	Message: ERREUR CLINIQUE : La date de décès est antérieure à la date de naissance.



In [4]:
print(" COMPARATIF : Validation par if/else")

patient_fhir_json = {
    "resourceType": "Patient",
    "birthDate": "1990-01-01",
    "deceasedDateTime": "1985-05-15"
}

if "deceasedDateTime" in patient_fhir_json and "birthDate" in patient_fhir_json:
    if patient_fhir_json["deceasedDateTime"] < patient_fhir_json["birthDate"]:
        print(" ERREUR : La date de décès est antérieure à la naissance (détecté par script en dur).")
        print("\n Conclusion : Le script 'en dur' fonctionne pour 1 règle, mais devient ingérable pour 5000 règles médicales. SHACL permet de séparer la logique de l'exécution.")

 COMPARATIF : Validation par if/else
 ERREUR : La date de décès est antérieure à la naissance (détecté par script en dur).

 Conclusion : Le script 'en dur' fonctionne pour 1 règle, mais devient ingérable pour 5000 règles médicales. SHACL permet de séparer la logique de l'exécution.
